# Repair Rate Analysis

Can we predict which returning cars need repair using car-level features (segment, brand),  
or is the repair share stable enough to use as a fixed multiplier?

In [ ]:
import pandas as pd
import numpy as np
import matplotlib

import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

events = pd.read_csv('../data/car_events.csv', parse_dates=['event_date'])
returns = events[events['event_type'] == 'arrival_return'].copy()
returns['month']   = returns['event_date'].dt.to_period('M')
returns['quarter'] = returns['event_date'].dt.to_period('Q')

print(f'Total arrival_return events: {len(returns):,}')
print(f'Overall repair rate: {returns["needs_repair"].mean():.3f}')

## 1. Repair rate by segment and brand

In [ ]:
seg = returns.groupby('car_segment').agg(
    total   = ('needs_repair', 'count'),
    repairs = ('needs_repair', 'sum')
).assign(rate=lambda x: x['repairs'] / x['total'])

brand = returns.groupby('car_brand').agg(
    total   = ('needs_repair', 'count'),
    repairs = ('needs_repair', 'sum')
).assign(rate=lambda x: x['repairs'] / x['total']).sort_values('total', ascending=False).head(10)

print('By segment:')
print(seg.round(3).to_string())
print(f'  Range: {seg["rate"].min():.3f} – {seg["rate"].max():.3f}  (spread: {seg["rate"].max()-seg["rate"].min():.3f})')
print()
print('By brand (top 10 by volume):')
print(brand.round(3).to_string())
print(f'  Range: {brand["rate"].min():.3f} – {brand["rate"].max():.3f}  (spread: {brand["rate"].max()-brand["rate"].min():.3f})')

## 2. Quarterly stability

In [ ]:
qtr = returns.groupby('quarter').agg(
    total   = ('needs_repair', 'count'),
    repairs = ('needs_repair', 'sum')
).assign(rate=lambda x: x['repairs'] / x['total'])

print('Quarterly repair rate:')
print(qtr.round(3).to_string())
print(f'  Mean: {qtr["rate"].mean():.3f}  Std: {qtr["rate"].std():.4f}  Range: {qtr["rate"].min():.3f}–{qtr["rate"].max():.3f}')

print()
print('By segment × quarter (checking for drift):')
sq = returns.groupby(['car_segment', 'quarter']).agg(
    total   = ('needs_repair', 'count'),
    repairs = ('needs_repair', 'sum')
).assign(rate=lambda x: x['repairs'] / x['total'])['rate'].unstack()
print(sq.round(3).to_string())

## 3. Compound-level repair rates

In [ ]:
compounds = pd.read_csv('../data/compounds.csv')

cpd_rate = returns.groupby('compound_id').agg(
    total   = ('needs_repair', 'count'),
    repairs = ('needs_repair', 'sum')
).assign(rate=lambda x: x['repairs'] / x['total'])
cpd_rate = cpd_rate.merge(compounds[['compound_id', 'city', 'repair_capable']], on='compound_id')

print('Repair rate per compound:')
print(cpd_rate[['city', 'repair_capable', 'total', 'repairs', 'rate']].sort_values('rate').round(3).to_string(index=False))
print(f'\n  Overall std across compounds: {cpd_rate["rate"].std():.4f}')
print(f'  Range: {cpd_rate["rate"].min():.3f} – {cpd_rate["rate"].max():.3f}')

## 4. Visualisation

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(14, 5))
fig.suptitle('Repair Rate Stability Analysis', fontsize=13, fontweight='bold')

overall = returns['needs_repair'].mean()

# Segment
ax = axes[0]
ax.bar(seg.index, seg['rate'], color='#1f77b4', alpha=0.8)
ax.axhline(overall, color='red', lw=1.5, ls='--', label=f'Overall {overall:.3f}')
ax.set_ylim(0, 0.35)
ax.set_ylabel('Repair rate')
ax.set_title('By segment')
ax.legend(fontsize=8)
for i, (idx, row) in enumerate(seg.iterrows()):
    ax.text(i, row['rate'] + 0.005, f"{row['rate']:.3f}", ha='center', fontsize=8)

# Quarter
ax = axes[1]
ax.plot(qtr.index.astype(str), qtr['rate'], marker='o', color='#2ca02c', lw=2)
ax.axhline(overall, color='red', lw=1.5, ls='--', label=f'Overall {overall:.3f}')
ax.fill_between(range(len(qtr)),
                [overall - qtr['rate'].std()] * len(qtr),
                [overall + qtr['rate'].std()] * len(qtr),
                alpha=0.1, color='red', label='±1 std')
ax.set_ylim(0.20, 0.30)
ax.set_ylabel('Repair rate')
ax.set_title('Quarterly trend')
ax.set_xticklabels(qtr.index.astype(str), rotation=40, ha='right', fontsize=8)
ax.legend(fontsize=8)

# Compound
ax = axes[2]
colors = ['#d62728' if r else '#1f77b4' for r in cpd_rate.sort_values('rate')['repair_capable']]
bars = ax.barh(cpd_rate.sort_values('rate')['city'], cpd_rate.sort_values('rate')['rate'],
               color=colors, alpha=0.8)
ax.axvline(overall, color='black', lw=1.5, ls='--', label=f'Overall {overall:.3f}')
ax.set_xlabel('Repair rate')
ax.set_title('By compound\n(red = repair-capable)')
ax.legend(fontsize=8)

plt.tight_layout()
plt.show()

## 5. Conclusion

In [ ]:
print('=' * 55)
print('  REPAIR RATE ANALYSIS — SUMMARY')
print('=' * 55)
print(f'  Overall rate:          {overall:.3f}  ({overall:.1%})')
print(f'  By segment range:      {seg["rate"].min():.3f} – {seg["rate"].max():.3f}  (spread {seg["rate"].max()-seg["rate"].min():.3f})')
print(f'  By brand range:        {brand["rate"].min():.3f} – {brand["rate"].max():.3f}  (spread {brand["rate"].max()-brand["rate"].min():.3f})')
print(f'  Quarterly std:         {qtr["rate"].std():.4f}')
print(f'  Compound std:          {cpd_rate["rate"].std():.4f}')
print()
print('  DECISION: use compound-level repair rate as fixed')
print('  multiplier on arrival_return forecast.')
print()
print('  repair_demand = arrival_return_forecast × compound_rate')
print()
print('  Compound repair rates (use in simulation):')
for _, row in cpd_rate.sort_values('compound_id').iterrows():
    print(f"    {row['city']:15s}: {row['rate']:.4f}")
print('=' * 55)